# Combine sync (run by hand, e.g. on Fridays)

Brings the latest work from the feature branches into `combine`, **only for the folders each branch owns** (`combine_sync/sync_config.json`).

**Modes (widget `MODE`)**
* `dry_run` (default): changes nothing. Writes a file-by-file manifest: what each branch would bring, what is kept as the combine version, and any conflict.
* `merge_local`: merges into the sync folder only (`REPO_PATH`), commits locally, does **not** push. Review the commits there before pushing.
* `merge_and_push`: same as `merge_local`, then pushes to `origin/combine`.

**Safety rules**
* Runs only in the dedicated sync Git folder (`REPO_PATH`), never in a folder people work in.
* Never deletes or changes a source branch. Only `combine` is written.
* `databricks-skills/`, `Archived Skills/` and `combine_sync/` are always kept as the combine version. The run stops if their content changes.
* A file outside a branch's `include` list, or in its `keep_combine` list, is reset to the combine version.
* A conflict in a file the branch owns stops that branch (it is skipped and reported). The other branches still run.
* The push is refused if someone pushed to `combine` during the run.

**Note:** a merge records the whole branch as merged. A file left out by the rules stays on its branch, and will not come back in a later run unless it changes again. Every run lists those files in the manifest (`action = KEEP_COMBINE / OUTSIDE_INCLUDE`).

In [0]:
import datetime as dt

DEFAULT_REPO = "/Workspace/Users/patricio.guledjian.ext@bayer.com/kerendia_nba_3_1_combine_sync"
DEFAULT_REPORT = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/combine_sync"

dbutils.widgets.dropdown("MODE", "dry_run", ["dry_run", "merge_local", "merge_and_push"])
dbutils.widgets.text("REPO_PATH", DEFAULT_REPO)          # dedicated sync Git folder, nobody works in it
dbutils.widgets.text("WEEK_START", "")                  # YYYY-MM-DD; empty = 7 days ago (only labels commits)
dbutils.widgets.text("BRANCHES", "")                    # comma list; empty = all enabled branches in the config
dbutils.widgets.text("REPORT_DIR", DEFAULT_REPORT)

MODE = dbutils.widgets.get("MODE").strip()
REPO_PATH = dbutils.widgets.get("REPO_PATH").strip().rstrip("/")
WEEK_START = dbutils.widgets.get("WEEK_START").strip() or (dt.date.today() - dt.timedelta(days=7)).isoformat()
ONLY_BRANCHES = [b.strip() for b in dbutils.widgets.get("BRANCHES").split(",") if b.strip()]
REPORT_DIR = dbutils.widgets.get("REPORT_DIR").strip().rstrip("/")
RUN_TS = dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")

assert MODE in ("dry_run", "merge_local", "merge_and_push"), MODE
print(f"MODE={MODE}  REPO_PATH={REPO_PATH}  WEEK_START={WEEK_START}  BRANCHES={ONLY_BRANCHES or 'all enabled'}  RUN_TS={RUN_TS}")

In [0]:
import subprocess, fnmatch, os, json

def git(*args, check=True):
    """Run git in REPO_PATH; return (returncode, stdout)."""
    p = subprocess.run(["git", "-c", "core.quotepath=off", *args], cwd=REPO_PATH, capture_output=True, text=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed ({p.returncode}):\n{p.stdout}\n{p.stderr}")
    return p.returncode, p.stdout

def matches(path, patterns):
    """A pattern ending with / is a folder prefix; anything else is a glob on the full path."""
    for pat in patterns:
        if pat.endswith("/"):
            if path.startswith(pat):
                return True
        elif fnmatch.fnmatchcase(path, pat):
            return True
    return False

def classify(path, rule, protected):
    if matches(path, protected):
        return "PROTECTED"            # always the combine version
    if matches(path, rule.get("keep_combine", [])):
        return "KEEP_COMBINE"         # combine owns this file / folder
    if matches(path, rule.get("include", [])):
        return "BRING"                # the branch owns it: take the branch change
    return "OUTSIDE_INCLUDE"          # not this branch's folder: keep the combine version

def tree_id(rev, path):
    rc, out = git("rev-parse", f"{rev}:{path.rstrip('/')}", check=False)
    return out.strip() if rc == 0 else None

def exists_in(rev, path):
    return git("cat-file", "-e", f"{rev}:{path}", check=False)[0] == 0

In [0]:
assert os.path.isdir(os.path.join(REPO_PATH, ".git")), f"{REPO_PATH} is not a CLI-enabled Git folder"
assert "combine_sync" in REPO_PATH, "REPO_PATH must be the dedicated sync folder (its name must contain 'combine_sync')"

# Config is always read from origin/<target>, never from a feature branch
git("fetch", "--prune", "origin")
CFG = json.loads(git("show", "origin/combine:combine_sync/sync_config.json")[1])
TARGET, REMOTE = CFG["target_branch"], CFG["remote"]
PROTECTED = CFG["protected_paths"]
T = f"{REMOTE}/{TARGET}"

# The sync folder must not hold anyone's work: stop if it has uncommitted changes
status = git("status", "--porcelain")[1].strip()
assert not status, f"Sync folder has uncommitted changes, clean it first:\n{status}"

# Every run starts from the remote combine (local commits from an earlier merge_local are dropped)
git("checkout", "-B", TARGET, T)
START_SHA = git("rev-parse", "HEAD")[1].strip()
PROTECTED_IDS = {p: tree_id("HEAD", p) for p in PROTECTED}

if not git("config", "user.email", check=False)[1].strip():
    me = spark.sql("select current_user()").first()[0]
    git("config", "user.email", me); git("config", "user.name", me)

rules = [r for r in CFG["branches"] if r.get("enabled") and (not ONLY_BRANCHES or r["branch"] in ONLY_BRANCHES)]
print(f"{TARGET} at {START_SHA[:8]}; branches to check: {[r['branch'] for r in rules]}")
print("Protected folders (tree ids):", {k: (v or 'missing')[:8] for k, v in PROTECTED_IDS.items()})

In [0]:
import pandas as pd

file_rows, branch_rows = [], []
for rule in rules:
    b = rule["branch"]; ref = f"{REMOTE}/{b}"
    if git("rev-parse", "--verify", "-q", ref, check=False)[0] != 0:
        branch_rows.append(dict(branch=b, status="BRANCH_NOT_FOUND")); continue
    mb = git("merge-base", T, ref)[1].strip()
    head = git("rev-parse", ref)[1].strip()
    log = [l.split("\t") for l in git("log", "--format=%h\t%cs\t%an\t%s", f"{mb}..{ref}")[1].splitlines() if l]
    if not log:
        branch_rows.append(dict(branch=b, head=head[:8], status="UP_TO_DATE", commits=0)); continue

    # Files the branch changed since it last met combine (renames split into delete + add)
    out = git("diff", "--name-status", "--no-renames", mb, ref)[1]
    changes = [l.split("\t", 1) for l in out.splitlines() if l]

    # Conflicts if merged now (read-only)
    rc, mt = git("merge-tree", "--write-tree", "--name-only", T, ref, check=False)
    conflicted = set()
    if rc == 1:
        for line in mt.splitlines()[1:]:
            if not line.strip(): break
            conflicted.add(line.strip())

    n = {"BRING": 0, "KEEP_COMBINE": 0, "OUTSIDE_INCLUDE": 0, "PROTECTED": 0}
    blocking = []
    for st, path in changes:
        action = classify(path, rule, PROTECTED)
        n[action] += 1
        conflict = path in conflicted
        if conflict and action == "BRING":
            blocking.append(path)
        file_rows.append(dict(branch=b, change={"A": "added", "M": "modified", "D": "deleted"}.get(st[0], st), path=path,
                              action=action, conflict=conflict,
                              resolution=("STOP: owned file conflicts" if conflict and action == "BRING"
                                          else "combine version kept" if conflict else "")))
    branch_rows.append(dict(branch=b, head=head[:8], merge_base=mb[:8], commits=len(log),
                            commits_this_week=sum(1 for c in log if c[1] >= WEEK_START),
                            last_commit=log[0][1], authors=", ".join(sorted({c[2] for c in log})),
                            bring=n["BRING"], keep_combine=n["KEEP_COMBINE"], outside_include=n["OUTSIDE_INCLUDE"],
                            protected=n["PROTECTED"], conflicts=len(conflicted), blocking_conflicts=len(blocking),
                            status="CONFLICT_STOP" if blocking else "READY"))

branches_df = pd.DataFrame(branch_rows)
files_df = pd.DataFrame(file_rows, columns=["branch", "change", "path", "action", "conflict", "resolution"])
display(branches_df)
display(files_df.sort_values(["branch", "action", "path"]))

In [0]:
run_dir = f"{REPORT_DIR}/{RUN_TS}_{MODE}"
os.makedirs(run_dir, exist_ok=True)
branches_df.to_csv(f"{run_dir}/branches.csv", index=False)
files_df.to_csv(f"{run_dir}/files.csv", index=False)
print("Manifest saved to", run_dir)

In [0]:
results = []
if MODE == "dry_run":
    print("dry_run: nothing merged. Review the manifest above, then run with MODE=merge_local or merge_and_push.")
else:
    for row in branch_rows:
        b = row["branch"]
        if row.get("status") != "READY":
            results.append(dict(branch=b, result=f"skipped ({row.get('status')})")); continue
        rule = next(r for r in rules if r["branch"] == b)
        ref = f"{REMOTE}/{b}"; before = git("rev-parse", "HEAD")[1].strip()
        git("merge", "--no-ff", "--no-commit", ref, check=False)   # rc 1 = conflicts, handled below

        touched = set(git("diff", "--name-only", "--no-renames", "--cached", "HEAD")[1].splitlines())
        touched |= set(git("diff", "--name-only", "--diff-filter=U")[1].splitlines())
        reverted = 0
        for path in sorted(p for p in touched if p):
            if classify(path, rule, PROTECTED) == "BRING":
                continue
            if exists_in("HEAD", path):
                git("checkout", "HEAD", "--", path)                  # back to the combine version
            else:
                git("rm", "-q", "-f", "--", path)                    # new file outside the branch's folders
            reverted += 1

        unmerged = [p for p in git("diff", "--name-only", "--diff-filter=U")[1].splitlines() if p]
        changed_protected = [p for p in PROTECTED if tree_id("HEAD", p) != PROTECTED_IDS[p] or
                             git("diff", "--cached", "--quiet", "HEAD", "--", p, check=False)[0] != 0]
        if unmerged or changed_protected:
            git("merge", "--abort", check=False)
            results.append(dict(branch=b, result=f"ABORTED unmerged={unmerged} protected_changed={changed_protected}")); continue

        brought = len([p for p in git("diff", "--name-only", "--cached", "HEAD")[1].splitlines() if p])
        msg = (f"Combine sync {RUN_TS}: merge {b} @ {row['head']} ({row['commits']} commits, {brought} files)\n\n"
               f"Only paths in sync_config.json 'include' were taken; {reverted} other changed files kept as the combine version.")
        git("commit", "-q", "-m", msg)
        results.append(dict(branch=b, result="MERGED", files_brought=brought, files_kept_combine=reverted,
                            commit=git("rev-parse", "--short", "HEAD")[1].strip()))

results_df = pd.DataFrame(results)
if len(results_df):
    display(results_df); results_df.to_csv(f"{run_dir}/results.csv", index=False)

In [0]:
if MODE != "dry_run":
    # 1. Protected folders identical to the start of the run
    for p, tid in PROTECTED_IDS.items():
        assert tree_id("HEAD", p) == tid, f"Protected path {p} changed during the sync, nothing pushed"
    # 2. Nothing outside the brought files changed vs the start
    final_changed = [p for p in git("diff", "--name-only", "--no-renames", START_SHA, "HEAD")[1].splitlines() if p]
    bring_paths = set(files_df.loc[files_df.action == "BRING", "path"])
    unexpected = [p for p in final_changed if p not in bring_paths]
    assert not unexpected, f"Unexpected changed files, nothing pushed: {unexpected[:20]}"
    print(f"Checks passed: {len(final_changed)} files changed vs {START_SHA[:8]}, protected folders unchanged.")

    has_new_commits = git("rev-parse", "HEAD")[1].strip() != START_SHA
    if MODE == "merge_and_push" and has_new_commits:
        git("fetch", REMOTE, TARGET)
        remote_now = git("rev-parse", T)[1].strip()
        assert remote_now == START_SHA, f"{T} moved during the run ({START_SHA[:8]} -> {remote_now[:8]}); rerun the sync"
        git("push", REMOTE, f"HEAD:{TARGET}")
        print(f"Pushed {TARGET}: {START_SHA[:8]} -> {git('rev-parse', '--short', 'HEAD')[1].strip()}")
    elif MODE == "merge_and_push":
        print("Nothing new to push.")
    elif MODE == "merge_local":
        print(f"merge_local: commits are only in {REPO_PATH} (branch {TARGET}); nothing pushed.")
    print(git("log", "--oneline", f"{START_SHA}..HEAD")[1])